## Stateless Logic Circuits --> Stateful Logic Circuits

- Up to this point, we have covered all **stateless operations** performed by a computer

- These are useful to a point, but we are still missing some important bits of a modern computer
    - Modern computers needs persistent memory, and stateless operations cannot provide this
    - We have seen have stateless operations handle inputs. But what exactly coordinates the input and output? 
    
- These 2 questions lead us to our next section in building up a running computer from scratch. To execute software, a computer needs:
    - **Memory**: To retain state across instructions 
    - **Clock**: To step through execution sequentially

- In this notebook, we will construct stateful operations from scratch!
    - **SR Latch & D Latch:** Primitive 1-bit memory cells built using feedback loops in NOR/NAND logic gates.
    - **D Flip-Flops:** Edge-triggered memory elements that only update state on clock transitions.
    - **Registers:** Parallel collections of flip-flops used to store multi-bit data words (e.g., 4-bit CPU registers).
    - **Register File & RAM:** Addressable arrays of registers built using Decoders and MUXes from Notebook 3.

### 1. Latches (1-Bit Bistable Memory)

#### Set Reset Latch

- Let's start by seeing how stateless operations, when combined in a a specific configuration, can let us retain the state of a bit!

- We start by illustrating the basic concept:
    - Instead of seeing "memory" as some magical state-retaining machine, the main insight is that persistent state can be obtained by feeding a sequence of stateless operations inputs in a loop!

- We'll start with the most basic example, the Set-Reset Latch (SR Latch) to illustrate this point

In [ ]:
from utils import *

def sr_latch(set: TRANSISTOR_OUTPUT, reset: TRANSISTOR_OUTPUT, val_prev: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    The purpose of an SR Latch is to allow us to persist the state of 1 bit of information. Let's call this `val`.
    
    To do this, we take in 3 inputs:
        - A `set` signal, indicating whether we should set `val` to 1
        - A `reset` signal, indicating whether we should set `val` to 0
        - The previous value held by `val`, which call `val_prev`
        - Note that in an SR Latch, `set` and `reset` must not both be 1 at the same time. 
    
    Using these, the SR latch outputs a value of `val`, and its inverse `val_prime`
        - Note that though `val_prime` is the inverse of `val`, it is not derived by passing `val` through a NOT gate
        - Instead, `val_prime` is computed separately, to ensure that there is no delay between the output `val` and `val_prime` being emitted
    
    The truth table below shows how the SR Latch turns the inputs into persistent byte value storage.
        - However, keep in mind that this doesn't always happen in one pass!
        - Sometimes, we can end up with intermediate val/val_prime values that require a few steps for its value to settle
        - We will show these in the truth table below

    Truth Table / Trace:
        - val_prev = 0 (val_prime_prev = 1):

            - set = 0, reset = 0 (HOLD STATE): Neither set nor reset. We want the circuit to output the same `val` it already has
                - Pass 1: val_prime = NOR(set=0, val_prev=0) = 1
                - Pass 1: val = NOR(reset=0, val_prime=1) = 0
                - Settled: (val=0, val_prime=1)

            - set = 0, reset = 1 (RESET STATE): Reset state. But since val = 0 already, we continue to output val = 0
                - Pass 1: val_prime = NOR(set=0, val_prev=0) = 1
                - Pass 1: val = NOR(reset=1, val_prime=1) = 0
                - Settled: (val=0, val_prime=1)

            - set = 1, reset = 0 (SET STATE): Set state.
                - Pass 1: val_prime = NOR(set=1, val_prev=0) = 0
                - Pass 1: val = NOR(reset=0, val_prime=0) = 1
                - +++++++++++++++++++++++++++++++++++++++++++++++++
                - Pass 2: val_prime = NOR(set=1, val=1) = 0
                - Pass 2: val = NOR(reset=0, val_prime=0) = 1
                - +++++++++++++++++++++++++++++++++++++++++++++++++                
                - Settled: (val=1, val_prime=0)

        - val_prev = 1 (val_prime_prev = 0):

            - set = 0, reset = 0 (HOLD STATE):
                - Pass 1: val_prime = NOR(set=0, val_prev=1) = 0
                - Pass 1: val = NOR(reset=0, val_prime=0) = 1
                - Settled: (val=1, val_prime=0)

            - set = 0, reset = 1 (RESET STATE):
                - Pass 1: val_prime = NOR(set=0, val_prev=1) = 0
                - Pass 1: val = NOR(reset=1, val_prime=0) = 0
                - +++++++++++++++++++++++++++++++++++++++++++++++++
                - Pass 2: val_prime = NOR(set=0, val=0) = 1
                - Pass 2: val = NOR(reset=1, val_prime=1) = 0
                - +++++++++++++++++++++++++++++++++++++++++++++++++
                - Settled: (val=0, val_prime=1)

            - set = 1, reset = 0 (SET STATE):
                - Pass 1: val_prime = NOR(set=1, val_prev=1) = 0
                - Pass 1: val = NOR(reset=0, val_prime=0) = 1
                - Settled: (val=1, val_prime=0)
    """

    '''
    if set == 1 and reset == 1:
        raise ValueError("Invalid State: 'set' and 'reset' cannot both be 1.")

    val = val_prev
    val_prime = 0 if val_prev == 1 else 1

    # Loop until the cross-coupled NOR gates stabilize into equilibrium
    prev_state = None
    while (val, val_prime) != prev_state:
        prev_state = (val, val_prime)

        # Gate 1: Drives val_prime using external 'set' and internal feedback 'val'
        val_prime = nor_gate(set, val)

        # Gate 2: Drives val using external 'reset' and internal feedback 'val_prime'
        val = nor_gate(reset, val_prime)

    return val, val_prime


#### D Latch

- The SR latch works, but notice its very annoying flaw. The moment you pass both set=1 and reset=1, the entire SR latch breaks. We simply cannot have this set of inputs
    - In a circuit board, there are many reasons why this sometimes happens. External interference, electrical leaks etc can all add up to reduce reliability
    - This means that any board using an SR latch is always 1 moment away from disaster. We never know when we may accidentally send (1,1) to the SR latch and cause a meltdown

- THe D Latch was invented to solve precisely this problem

In [ ]:
def d_latch(setval: TRANSISTOR_OUTPUT, enable: TRANSISTOR_OUTPUT, val_prev: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    Similar to SR Latch, the D latch is designed to handle the persistence of 1 bit of data. 
    In fact, the D latch LITERALLY contains an SR Latch, just with better UI to avoid mistakes with the inputs.

    Inputs   
        - `setval`
            - Unlike the SR latch, which uses 1 input for `setval` and 1 for `reset`, the D Latch only takes 1 input for both. 
            - If the input is 1, it signals `setval`. Else, it signals `reset`
            - This avoids the possibility of having both setval=1 and reset=1, which was the problem with the SR latch setup
        - `enable`
            - An extra gate that determines whether the `setval` input is passed foward
            - If `enable = 0`, `setval` is always ignored, no matter its value
        - `val_prev`
            - `val` from previous time step
    '''
    reset = cmos_NOT(setval)
    
    setval = cmos_AND(setval, enable)
    reset = cmos_AND(reset, enable)
    
    return sr_latch(setval, reset, val_prev)

##### Applications of D-Latch

- In computers, D-Latches are used in both logic processing and in memory
    - Note that by "memory", I mean that D-Latches are used to form 6-transistor storage cells of Static RAM (SRAM), which are used in high-speed CPU caches, rather than Dynamic RAM (DRAM, which relies on leaking micro-capacitors) or persistent storage (Flash/SSDs)

- In both these use cases, they are used in a slightly different way. We will discuss the details of both use cases here to gain a better appreciation of why D-latches are necessary

##### D Latch in Logic Processing

- In computational logic a D-Latch acts as a timed firewall and shield for chained pipelines (e.g. I want to connect an Adder to a Multiplier):
    - **Shielding Intermediate Noise:** 
        - In a computer, all computations take a step when the CPU sends a tick signal
        - In between tick signals, values can fluctuate as the current flows through the circuit
        - Suppose we want to compose an ADDER with a MULTIPLIER. This fluctuating signal can cause the ADDER's output to fluctuate, which causes the MULTIPLIER's output to fluctutate
        - Downstream circuits may capture these values erroneously, which affects their processing
        - By placing a D-Latch between the ADDER and MULTIPLIER, and passing the CPU tick as a `enable` signal to the D-Latch, no value is captured by the D-Latch until the signal has settled. 
        - Therefore, the multiplier never receives garbaage input
    - **Pipeline Isolation:** 
        - The D Latch holds the settled result steady, so the next processing stage MULTIPLIER works cleanly on the current data without noise, while the ADDER can  immediately begin computing the next set of inputs in parallel

- Recall from our function above that a D Latch takes in 3 inputs; `enable`, `set`, and `val_prev`. So in logic processing: 
    - `enable` takes its value from the system clock wire, which ensure that the D Latch only stores the value when the system clock actually ticks, and so it never accidentally stores intermediate noise 
    - `set` takes in value from its own dedicated wire connected to the upstream operator (e.g. `ADDER`)
    - `val_prev` is the value that's already in the D Latch

##### D-Latch in Cache Memory (SRAM Grid Arrays)

- In memory architecture, D-Latches are arranged into a 2D grid to form Static RAM (SRAM), which is the ultra-fast memory used for CPU caches (L1, L2, L3 caches)

- While main system RAM is also called "RAM", it is called DRAM, which replaces transistor D-Latches with microscopic capacitors. SRAM uses true D-Latches.

- We will handle this in greater detail in the section on `Register File & RAM` below. For now, treat this as an introductory overview

- In memory architecture, D-Latches are arranged into a 2D grid to form **Static RAM (SRAM)**, which is the ultra-fast memory used for CPU caches ($L1, L2, L3$).

- While main system RAM is also called "RAM", it uses **Dynamic RAM (DRAM)**, which replaces transistor D-Latches with microscopic capacitors that require periodic refreshing. SRAM uses true transistor-based D-Latches to achieve maximum speed.

- **The Grid Architecture (Word Lines & Bit Lines):**
    - Memory cells are organized in rows and columns to share control wires and minimize total silicon wiring:
        - **Word Line (Horizontal):** Connected directly to the `enable` pin ($E$) of every D-Latch in that row. Driven by the Address Decoder to unlock an entire row at once.
        - **Bit Line (Vertical):** Connected directly to the `setval` pin ($D$) of every D-Latch in that column. Carries data bits to or from the CPU bus.

- **Recall our 3 inputs for the D-Latch in SRAM Memory:**
    - `enable`: Takes its signal from the horizontal **Word Line**. When `Word Line = 1`, all 64 cells in that row open their enable gates.
    - `setval`: Takes its signal from the vertical **Bit Line**. Supplies the raw data bit (`1` or `0`) to be written into the cell.
    - `val_prev`: Represented by the **internal physical transistor feedback loop** inside the memory cell, holding the bit indefinitely as long as power is applied.

- **Column Masking:**
    - Because a single Word Line enables all 64 latches across a row simultaneously ($E = 1$), updating only a single byte requires **Column Masking**.
    - The memory controller actively drives the target Bit Lines while placing non-targeted Bit Lines into a neutral state, allowing those unselected cells to preserve their internal `val_prev` state even while their Enable door is open.

- *(We will explore the complete multi-row layout and address decoding logic in detail in Section 4 on `Register File & RAM`).*

### 2. Edge-Triggered D Flip-Flop

- We have seen how D Latches work. Let's zoom in on the portion where we said that D Latches sit between operators, wired to the computer's clock signal line 
    - Suppose this pipeline: `ADDER --> D Latch --> MULTIPLIER`
    - We said that by wiring D Latch to the computer's clock, it only accepts and stores the value from ADDER when the clock ticks
    - This prevents the D Latch from storing intermediate noise

- Notice that this is only one half of the problem. Yes, we have prevented noise from ADDER from hitting D Latch until the next clock step. But once D Latch stores a value, it IMMEDIATELY flows to MULTIPLIER, no matter what multiplier is meant to do. 
    - What we want: 
        - t0: adder operation 
        - t1: D Latch accepts and stores value A, Multiplier working on previous D Latch value. Adder starts working on new value B
        - t2: D Latch passes new value A onward to Multiplier, and accepts new value B from ADDER
    - What we get with a pure D Latch
        - t0: adder operation 
        - t1: D Latch accepts and stores new value A, Multiplier imemdiately gets new value A
        - t2: D Latch accepts and stores new value B, Multiplier imemdiately gets new value B
    - Multiplier never has a "clock tick" of time to work on new value A!

- This is why a pure D Latch is not enough. We need something that sits between ADDER and D Latch, but we also need something that sits between D Latch and MULTIPLIER!
    - Basically, we need a mechanism to ensure that we only ever advance 1 operation per clock tick

- This is where an **Edge-Triggered D Flip Flop** comes in


#### How does it work?

- The idea is quite simple. Instead of a pure D Latch, we combine 2 D latches back to back!
    - The clock wire is still connected to both D latches, but one of them has a NOT modifier
    - When clock = 0: 
        - The first (master) latch is enabled, and it takes in the value from its upstream operator
        - The second (slave) latch is disabled, so nothing can reach it. It stably holds the value it was provided and passes it to the downstream operator
    - When clock = 1:
        - The first (master) latch is disabled, and it blocks any new value from input D
        - The second (slave) latch is enabled, and takes in the value from the master D latch, and passes it downstream

```
                            D FLIP-FLOP
           +-------------------------------------------+
           |                                           |
           |   +---------------+   +---------------+   |
Input D --->---| Master Latch  |---|  Slave Latch  |---|---> Output Q
           |   |   (E = CLK')  |   |   (E = CLK)   |   |
           |   +---------------+   +---------------+   |
           +-------------------------------------------+
                     ^                   ^
                     |      +-----+      |
CLK -----------------+------| NOT |------+
                            +-----+
```

In [ ]:
def d_flip_flop(input: TRANSISTOR_OUTPUT, enable: TRANSISTOR_OUTPUT, slave_val_prev: TRANSISTOR_OUTPUT, master_val_prev: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    Rising-Edge Triggered Master-Slave D Flip-Flop.
    
    Inputs:
        - input Input data bit
        - enable: From system clock signal
        - slave_val_prev: Stored output Q (from Slave)
        - master_val_prev: Stored intermediate state Q (from Master)
        
    Returns:
        - (Q, Q_bar, master_Q): Updated output Q, inverse Q_bar, and new internal Master Q state
    '''
    not_enable = cmos_NOT(enable)
    
    # Stage 1: Master Latch (Enabled when Clock is LOW)
    master_q, _ = d_latch(input, not_enable, master_val_prev)
    
    # Stage 2: Slave Latch (Enabled when Clock is HIGH)
    slave_q, not_slave_q = d_latch(master_q, enable, slave_val_prev)
    
    return slave_q, not_slave_q, master_q

### 3. N-Bit Registers

- We have seen how a single **Edge-Triggered D Flip-Flop** acts as a 1-bit memory airlock for CPU pipelines.

- In modern computing, CPUs operate on multi-bit chunks of data simultaneously (e.g., 8-bit, 32-bit, or 64-bit words). A **Register** is simply an array of $N$ D Flip-Flops connected in parallel, sharing a common clock and control signals.

#### Building an N-Bit Register

- To build an 8-bit register, we line up 8 individual D Flip-Flops side-by-side.

- **Data Bus ($D_0 \dots D_7$):** Each flip-flop receives 1 bit from the 8-bit input bus
    - This is equivalent to the `input` field that each flip flop receives from before

- **Clock Line ($CLK$):** A single system clock wire is routed to the `enable` pin of **all 8 flip-flops simultaneously**. On the rising edge ($0 \rightarrow 1$), all 8 bits are captured in exact synchronization.
    - This is equivalent to the `enable` field that each flip flop receives from before

```
                                     8-BIT REGISTER
        +-----------------------------------------------------------------------+
        |                                                                       |
Data    |   D[0]           D[1]           D[2]                  D[7]            |    Outputs
Bus ====|====|==============|==============|=====================|==============|===> Q[0..7]
D[0..7] |    |              |              |                     |              |
        |    v              v              v                     v              |
        |  +---+          +---+          +---+                 +---+            |
        |  | D |          | D |          | D |                 | D |            |
        |  |   |--> Q[0]  |   |--> Q[1]  |   |--> Q[2]  ...    |   |--> Q[7]    |
        |  |   |          |   |          |   |                 |   |            |
        |  |DFF0          |DFF1          |DFF2                 |DFF7            |
        |  +---+          +---+          +---+                 +---+            |
        |    ^              ^              ^                     ^              |
        |    |              |              |                     |              |
        +----|--------------|--------------|---------------------|--------------+
             |              |              |                     |
CLK ---------+--------------+--------------+---------------------+
             (Global Clock Line connected to all 8 Enable pins)

```

- From the schematic above, we see that instead of a single `CLK` input, we now have 8 of the same inputs from `CLK` to the 8 flip flops

- The data bus passes one bit of data per flip flop, which serves its job of accepting new data with every clock tick

##### Complication: What if operations take longer than 1 clock tick?

- Notice one problem here: we have no way to control when a flipflop should or should not accept data from the data bus! Once the clock ticks, the over in the flip flop is overwritten, no matter what

- This works if all downstream operations can complete within 1 clock tick. However, there are definitely operations that need more than 1 clock tick to complete!
    - If we stick to the previous schematic, those operations would be impossible, because they will be interrupted with every tick of the system clock signal

- To get around this, a register needs something to give it finer control of every flip flop; a bit that tells it whether or not the downstream operation has completed. This is known as the **LOAD** line
    - This goes beyond the `enable` signal from the clock line
    - The clock line the register that it CAN write the next operation's value, the load line tells that register that it SHOULD write the next operation's value

- Think of it as an **AND** operation. We need both clock `enable` and load `enable` before the register records the value

- This is a schematic for one such flip flop that takes in both the load and clock lines
    - Every D flipflop is paired with a `MUX`, which takes in the flip flop's existing value, and a data bus value
	- Depending on the LOAD signal, it outputs one to the other to the flip flop

```
                       +---------------+
New Data Bus --------> | 1             |
                       |     MUX       |-----> [ D Flip-Flop ]
Existing Output (Q) -> | 0             |              ^
                       +---------------+              |
                               ^                      |
                               |                      |
LOAD Signal -------------------+                      |
                                                      |
System Clock (CLK) -----------------------------------+
```

- Extending this to a register of 8 bits:

```
									   8-BIT REGISTER WITH LOAD ENABLE
        +-----------------------------------------------------------------------------------------------+
        |                                                                                               |
Data    |   D[0]               D[1]                               D[7]                                  |   Outputs
Bus ====|====|==================|==================================|====================================|==> Q[0..7]
D[0..7] |    |  +--------+      |  +--------+                      |  +--------+                        |
        |    |  |        |      |  |        |                      |  |        |                        |
        |    +->|1       |      +->|1       |                      +->|1       |                        |
        |       |  MUX0  |--+      |  MUX1  |--+                      |  MUX7  |--+                     |
        |  +--->|0       |  |  +-->|0       |  |                 +--->|0       |  |                     |
        |  |    +--------+  |  |   +--------+  |                 |    +--------+  |                     |
        |  |        ^       |  |       ^       |                 |        ^       |                     |
        |  |        | (L)   |  |       | (L)   |                 |        | (L)   |                     |
LOAD ---|--|--------+-------|---|------+-------|-----------------|--------+-------|---------------------+
        |  |                v   |              v                 |                v                     |
        |  |              +---+ |            +---+               |              +---+                   |
        |  |              | D | |            | D |               |              | D |                   |
        |  +--------------|   |<+------------|   |<--------------+--------------|   |<---+              |
        |  (Q0)           |DFF0| (Q1)        |DFF1|               (Q7)          |DFF7|   |              |
        |                 +---+              +---+                              +---+    |              |
        |                   ^                  ^                                  ^      |              |
        |                   | (C)              | (C)                              | (C)  |              |
CLK ----|-------------------+------------------+----------------------------------+------|--------------+
        |                                                                                |
        +--------------------------------------------------------------------------------+
```

#### Python Implementation

In [ ]:
def n_bit_register(
    inputs: list[TRANSISTOR_OUTPUT], 
    enable: TRANSISTOR_OUTPUT, 
    load: TRANSISTOR_OUTPUT, 
    slave_vals_prev: list[TRANSISTOR_OUTPUT], 
    master_vals_prev: list[TRANSISTOR_OUTPUT]
) -> tuple[list[TRANSISTOR_OUTPUT], list[TRANSISTOR_OUTPUT], list[TRANSISTOR_OUTPUT]]:
    '''
    N-Bit Synchronous Register with Load Enable using Master-Slave D Flip-Flops.
    
    Inputs:
        - inputs: List of N input data bits
        - enable: System clock signal
        - load: Control signal (1 = load new input on tick, 0 = hold current Q)
        - slave_vals_prev: List of N stored output bits (from Slaves)
        - master_vals_prev: List of N stored intermediate bits (from Masters)
        
    Returns:
        - (outputs, not_outputs, new_master_states): Tuple of outputs Q, ~Q, and internal Master states
    '''
    n = len(inputs)
    outputs = [0] * n
    not_outputs = [0] * n
    new_master_states = [0] * n
    
    for i in range(n):
        # MUX selects between external input bus (load=1) and recirculating Q (load=0)
        d_in = cmos_MUX(select=load, in0=slave_vals_prev[i], in1=inputs[i])
        
        # Execute Master-Slave D Flip-Flop
        slave_q, not_slave_q, master_q = d_flip_flop(
            input=d_in, 
            enable=enable, 
            slave_val_prev=slave_vals_prev[i], 
            master_val_prev=master_vals_prev[i]
        )
        
        outputs[i] = slave_q
        not_outputs[i] = not_slave_q
        new_master_states[i] = master_q
        
    return outputs, not_outputs, new_master_states

### 4. Register File & RAM (Addressable Memory)

#### Basics

- We now know how to build an **N-Bit Register** that stores a word of data safely using a $LOAD$ signal.
- However, CPUs contain multiple registers (e.g., $R_0, R_1, R_2, R_3$). To manipulate specific registers without running individual control lines to every single one, we group them into an **Addressable Memory Array** (a Register File or SRAM Grid).

```
                          REGISTER FILE (4 registers x N bits)

          +--------------------------------------------------------------------------+
          |                                                                          |
Write     |  D[0..N-1] (shared write data bus)                                       |
Data  ====|===========================+                                              |
          |                           | (to D input of R0..R3)                       |
          |                           v                                              |
          |  +----------+    +-----------------+          +------------------+       |
Write     |  |          |-W0>| Register 0 (R0) |== Q0 ===>|                  |       |
Address --|->|  2-to-4  |-W1>| Register 1 (R1) |== Q1 ===>|  4-to-1 N-bit    |=======|==> Read Data Q[0..N-1]
[0..1]    |  | decoder  |-W2>| Register 2 (R2) |== Q2 ===>|  MUX             |       |
          |  |          |-W3>| Register 3 (R3) |== Q3 ===>|  (read select)   |       |
          |  +----------+    +-----------------+          +------------------+       |
          |       ^                   ^                            ^                 |
LOAD -----|-------+ (decoder enable)  |                            |                 |
          |                           |                            |                 |
Read      |                           |                            |                 |
Address --|---------------------------|----------------------------+ (selects Qi)    |
[0..1]    |                           |                                              |
CLK ------|---------------------------+ (to every flip-flop in R0..R3)               |
          |                                                                          |
          +--------------------------------------------------------------------------+
```

#### Address Decoding & Data Routing

- To make an array of registers addressable, we introduce two key combinational logic blocks:
	- **Decoder (Address Decoder):** Takes a $k$-bit binary address and activates exactly **one** output line out of $2^k$ rows (the **Word Line**)
	- **Multiplexer (Output Selector):** Takes the data outputs from all registers and routes the requested register's output onto the main Read Bus.

- **Write Mechanics:** 
  - To write to Register 2, the CPU sets `Write Address = 02`. 
  - The Address Decoder sets $W_2 = 1$ while keeping $W_0, W_1, W_3 = 0$. 
  - On the rising clock edge, **only Register 2** updates its D Flip-Flops; all other registers recirculate their internal $Q$ values.

- **Read Mechanics:**
  - Reading does **not** depend on the clock tick (it is asynchronous/combinational).
  - Setting `Read Address = 01` tells the output MUX to immediately route `Q1` directly to the Read Data Bus.

#### Python Implementation

In [ ]:
def register_file_4xN(
    write_data: list[TRANSISTOR_OUTPUT],
    write_addr: list[TRANSISTOR_OUTPUT],  # 2-bit address [A0, A1]
    read_addr: list[TRANSISTOR_OUTPUT],   # 2-bit address [A0, A1]
    write_enable: TRANSISTOR_OUTPUT,      # Global LOAD signal
    enable: TRANSISTOR_OUTPUT,            # System Clock (CLK)
    slave_state_matrix: list[list[TRANSISTOR_OUTPUT]],   # 4 x N stored outputs
    master_state_matrix: list[list[TRANSISTOR_OUTPUT]]   # 4 x N intermediate states
) -> tuple[list[TRANSISTOR_OUTPUT], list[list[TRANSISTOR_OUTPUT]], list[list[TRANSISTOR_OUTPUT]]]:
    '''
    4-Register Addressable Register File (4 words of N-bits each).
    
    Inputs:
        - write_data: N-bit input data bus
        - write_addr: 2-bit binary write selection
        - read_addr: 2-bit binary read selection
        - write_enable: Master LOAD line
        - enable: System clock signal
        - slave_state_matrix: 4xN array holding stored outputs for all 4 registers
        - master_state_matrix: 4xN array holding intermediate states for all 4 registers
        
    Returns:
        - (read_data, new_slave_matrix, new_master_matrix): Currently selected register output and updated memory state
    '''
    # Step 1: Decode 2-bit Write Address into 4 Word Lines (W0..W3)
    word_lines = cmos_DECODER_2to4(write_addr)
    
    new_slave_matrix = []
    new_master_matrix = []
    reg_outputs = []
    
    # Step 2: Process all 4 Registers in Parallel
    for r in range(4):
        # Register r receives LOAD = 1 ONLY if write_enable is active AND its Word Line is active
        reg_load = cmos_AND(write_enable, word_lines[r])
        
        # Execute N-Bit Register for Row r
        outputs, _, master_states = n_bit_register(
            inputs=write_data,
            enable=enable,
            load=reg_load,
            slave_vals_prev=slave_state_matrix[r],
            master_vals_prev=master_state_matrix[r]
        )
        
        reg_outputs.append(outputs)
        new_slave_matrix.append(outputs)
        new_master_matrix.append(master_states)
        
    # Step 3: Route Selected Register Output to Read Bus via MUX
    read_data = cmos_MUX_4to1(
        select=read_addr,
        in0=reg_outputs[0],
        in1=reg_outputs[1],
        in2=reg_outputs[2],
        in3=reg_outputs[3]
    )
    
    return read_data, new_slave_matrix, new_master_matrix